# Đánh giá luồng đặt vé sau Pha 2

Dùng Dataset riêng chứa bundle nguồn của vòng cải tiến. Chọn **Accelerator: None** và **Internet: On**. Chạy theo thứ tự trong cùng phiên Kaggle. Bộ 144 lượt Pha 2 nằm ở notebook riêng.

D0 là cấu hình D đối chứng. D1 tách session xác thực. D2 kết hợp D1 và giới hạn 4 yêu cầu đặt vé mỗi worker. Mọi biến thể dùng chu kỳ tối thiểu 100 ms mỗi người dùng ảo để bộ tạo tải không bị phản hồi 503 làm bão hòa.


In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, tarfile, tempfile


def source_roots(directory):
    required = ['app/main.py', 'alembic.ini', 'requirements.txt',
                'experiments/phase2/bootstrap.py', 'experiments/phase2/requirements.lock.txt',
                'app/api/booking_admission.py']
    return sorted({path.parents[2] for path in directory.rglob('bootstrap.py')
                   if all((path.parents[2] / name).is_file() for name in required)})


def prepare_source(input_root, working_root):
    input_root, working_root = Path(input_root), Path(working_root)
    archives = sorted(input_root.rglob('cinema-phase2-source.tar.gz'))
    if len(archives) > 1:
        raise RuntimeError('Có nhiều gói source; chỉ gắn một Dataset: ' + str(archives))
    if archives:
        source = archives[0]
        checksum = hashlib.sha256(source.read_bytes()).hexdigest()
        mode = 'Tệp nén'
    else:
        roots = source_roots(input_root)
        if len(roots) != 1:
            raise RuntimeError('Cần đúng một bộ source của vòng đặt vé. Chọn Add Input để gắn Dataset '
                               'chứa gói source hoặc source đã giải nén. Các bộ tìm thấy: ' + str(roots))
        source = roots[0]
        digest = hashlib.sha256()
        for path in sorted(source.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts and path.suffix != '.pyc':
                digest.update(path.relative_to(source).as_posix().encode() + b'\0')
                digest.update(hashlib.sha256(path.read_bytes()).digest())
        checksum = digest.hexdigest()
        mode = 'Source đã được Kaggle giải nén'
    working_root.mkdir(parents=True, exist_ok=True)
    repo = working_root / ('cinema-phase2-' + checksum[:12])
    if not repo.exists():
        # Publish only a complete copy; an interrupted cell can be run again.
        with tempfile.TemporaryDirectory(prefix='phase2-source-', dir=working_root) as temporary:
            staging = Path(temporary) / 'source'
            if archives:
                staging.mkdir()
                with tarfile.open(source) as archive:
                    archive.extractall(staging, filter='data')
            else:
                shutil.copytree(source, staging, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
            roots = source_roots(staging)
            if len(roots) != 1:
                raise RuntimeError('Gói source thiếu code vòng đặt vé. Tạo lại bundle từ nhánh cải tiến: ' + str(roots))
            roots[0].rename(repo)
    if repo not in source_roots(repo):
        raise RuntimeError('Thư mục working chưa có source đầy đủ: ' + str(repo))
    print('Input mode:', mode)
    print('Input source:', source)
    print('Source checksum:', checksum)
    print('Repository:', repo)
    return repo


repo = prepare_source('/kaggle/input', '/kaggle/working')


## Chuẩn bị môi trường

Ô này cài PostgreSQL 16, Redis 7 và môi trường Python 3.12 với dependency đã khóa. Trình cài chỉ chạy trong Kaggle; các dịch vụ thử nghiệm được khởi động riêng khi kiểm chứng hoặc đo tải.


In [ ]:
# Export figures to files using a backend available in the experiment environment.
os.environ['MPLBACKEND'] = 'Agg'
sys.path.insert(0, str(repo))
from experiments.phase2.notebook import run_checked
run_checked([sys.executable, str(repo / 'experiments/phase2/bootstrap.py'), '--install'], cwd=repo)
python = '/kaggle/working/phase2-venv/bin/python'
def run(*arguments):
    run_checked([python, *arguments], cwd=repo)


## Kiểm chứng chức năng

Chạy toàn bộ kiểm thử, bao gồm PostgreSQL qua migration, cache fallback và tranh chấp giữa hai worker. Dừng tại đây nếu kiểm thử không đạt.


In [ ]:
run('-m', 'experiments.phase2.verify')


## Chẩn đoán kết nối (tùy chọn)

Ô này đo D0 tại 50 và 100 người dùng trong R2, R3 và R4. Xem `booking-timing.csv` trước khi chọn so sánh. Có thể bỏ qua nếu đã có dữ liệu chẩn đoán phù hợp.


In [ ]:
diagnostic = Path('/kaggle/working/booking-diagnostic')
run('-m', 'experiments.phase2.runner', '--profile', 'booking-diagnostic', '--output-root', str(diagnostic))
run('-m', 'experiments.phase2.report', str(diagnostic))


## Kiểm tra nhanh

Smoke chạy 9 lượt cho ba biến thể hoặc 6 lượt nếu chỉ thử lớp giới hạn. Nếu cần chạy lại, dùng một thư mục đầu ra mới.


In [ ]:
# Chọn phép so sánh trước khi chạy smoke và giữ nguyên cho lượt full.
compare_split_auth = True
comparison_options = () if compare_split_auth else ('--without-split-auth',)
smoke = Path('/kaggle/working/booking-followup-smoke')
run('-m', 'experiments.phase2.runner', '--profile', 'booking-smoke',
    '--output-root', str(smoke), *comparison_options)
run('-m', 'experiments.phase2.report', str(smoke))


## Đo chính

So sánh D0, D1 và D2 trong 81 lượt. Nếu chọn `compare_split_auth = False`, so sánh D0 và D-limit trong 54 lượt. Dự kiến khoảng 1–2 giờ tùy môi trường. Chỉ chạy tiếp khi smoke đạt. Dùng cùng phần cứng, source và cấu hình trong toàn bộ lượt đo.


In [ ]:
import json

smoke_status = json.loads((smoke / 'summary.json').read_text())
expected_smoke = 9 if compare_split_auth else 6
if (not smoke_status['complete'] or not smoke_status['canonical']
        or smoke_status['profile'] != 'booking-smoke' or smoke_status['expected_runs'] != expected_smoke):
    raise RuntimeError('Smoke phải đo đủ cấu hình của phép so sánh đã chọn.')
full = Path('/kaggle/working/booking-followup-results')
run('-m', 'experiments.phase2.runner', '--profile', 'booking-followup',
    '--output-root', str(full), *comparison_options)


## Tổng hợp và tải kết quả

Tải `booking-followup-results.tar.gz` từ Output của Kaggle. Kiểm tra `booking_followup_complete`. Nếu lượt đo bị gián đoạn, báo cáo vẫn giữ các lượt đã có và trạng thái chưa hoàn tất. Giữ archive Pha 2 và archive vòng đặt vé riêng.


In [ ]:
run('-m', 'experiments.phase2.report', str(full))
summary = json.loads((full / 'summary.json').read_text())
print({key: summary[key] for key in ['expected_runs', 'valid_runs', 'invalid_runs', 'booking_followup_complete']})
shutil.make_archive(str(full), 'gztar', root_dir=full.parent, base_dir=full.name)
